# 09. 텍스트 전처리 — 모델에 넣기 전에

노트북 08에서 문장을 `Tokenizer`로 숫자로 바꿨다. 그런데 그 전에 할 일이 더 있다.

실제 텍스트는 지저분하다:
- `love`, `loves`, `loving`, `loved` — 다 같은 뜻인데 **다른 단어로 취급**된다
- `the`, `is`, `a` 같은 단어는 너무 흔해서 **의미가 거의 없다**
- 한국어는 "먹었다", "먹는", "먹고" — **형태가 계속 바뀐다**

이런 걸 정리하는 게 **전처리(preprocessing)** 다. 실전 NLP는 이 전처리가 절반이다.

이 노트북에서 배우는 것:
1. **표제어 추출(Lemmatization)** — 단어를 사전 기본형으로 (`loves` → `love`)
2. **어간 추출(Stemming)** — 규칙으로 어미를 자르기 (`loves` → `lov`)
3. **불용어(Stopwords)** — 의미 없는 흔한 단어 제거
4. **한국어 형태소 분석** — KoNLPy로 한국어를 쪼개기
5. **실습** — 헌법 텍스트에서 명사 Top 10 뽑기

> 이 노트북은 `nltk`, `konlpy` 라이브러리를 쓴다. Colab에서 자동 설치·다운로드된다.


## 1부. 영어 전처리

### 1-1. 표제어 추출 (Lemmatization)

**표제어(lemma)** = 사전에 실리는 기본형. `lives` → `life`, `dies` → `die`.

`WordNetLemmatizer`는 **사전(WordNet)에 기반**해서, 실제로 존재하는 단어로 되돌린다.
어간 추출(다음)과 달리 **결과가 항상 진짜 단어**다.

준비: 여러 형태의 단어 목록.


In [1]:
words = ['policy', 'doing', 'organization', 'have', 'going', 'love',
         'lives', 'fly', 'dies', 'watched', 'has', 'starting']
print(words)


['policy', 'doing', 'organization', 'have', 'going', 'love', 'lives', 'fly', 'dies', 'watched', 'has', 'starting']


### 1-2. 품사(POS)를 알려주면 더 정확하다

`lemmatize`는 **기본적으로 명사로 가정**한다. 그래서 동사를 그냥 넣으면 제대로 안 된다:
- `dies` → `dy` (명사로 보고 잘못 처리)

품사를 `'v'`(동사)로 알려주면:
- `dies` → `die` (정확!)

`nltk.download('wordnet')`으로 WordNet 사전을 먼저 받는다.


In [2]:
import nltk
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')

wnl = WordNetLemmatizer()

# 품사 없이 (명사로 가정)
print("품사 없이:", [wnl.lemmatize(wd) for wd in words])

# 동사('v')로 지정
print("동사로   :", [wnl.lemmatize(wd, 'v') for wd in words])


[nltk_data] Downloading package wordnet to /root/nltk_data...


품사 없이: ['policy', 'doing', 'organization', 'have', 'going', 'love', 'life', 'fly', 'dy', 'watched', 'ha', 'starting']
동사로   : ['policy', 'do', 'organization', 'have', 'go', 'love', 'live', 'fly', 'die', 'watch', 'have', 'start']


### 1-3. 어간 추출 (Stemming) — 규칙으로 자르기

**어간 추출**은 사전을 안 쓴다. 단순히 **어미를 규칙대로 잘라낸다.** 빠르지만 거칠다.
결과가 **진짜 단어가 아닐 수도** 있다 (`policy` → `polici`).

두 가지 유명한 알고리즘을 비교한다:
- **Porter** : 비교적 보수적 (`lives` → `live`)
- **Lancaster** : 더 공격적으로 자름 (`organization` → `org`)

**표제어 추출 vs 어간 추출:**
- 표제어(Lemma) : 사전 기반, 느림, 결과가 진짜 단어 ← 정확도 중요할 때
- 어간(Stem) : 규칙 기반, 빠름, 결과가 조각일 수 있음 ← 속도·대량 처리


In [3]:
from nltk.stem import PorterStemmer, LancasterStemmer

print("원본     :", words)
p = PorterStemmer()
print("Porter   :", [p.stem(wd) for wd in words])
l = LancasterStemmer()
print("Lancaster:", [l.stem(wd) for wd in words])


원본     : ['policy', 'doing', 'organization', 'have', 'going', 'love', 'lives', 'fly', 'dies', 'watched', 'has', 'starting']
Porter   : ['polici', 'do', 'organ', 'have', 'go', 'love', 'live', 'fli', 'die', 'watch', 'ha', 'start']
Lancaster: ['policy', 'doing', 'org', 'hav', 'going', 'lov', 'liv', 'fly', 'die', 'watch', 'has', 'start']


### 1-4. 불용어(Stopwords) 제거 + 토큰화

**불용어** = `the`, `is`, `a`, `an` 처럼 너무 흔해서 **의미 판별에 도움이 안 되는** 단어.
NLP에서 보통 제거한다 (검색·분류에서 노이즈만 늘리므로).

**토큰화(Tokenization)** = 문장을 단어 단위로 쪼개는 것. `word_tokenize`가 문장부호까지 분리해준다.

```
"Family is not an important thing. It's everything."
  → ['Family', 'is', 'not', 'an', 'important', 'thing', '.', ...]
  → 불용어(is, an, ...) 제거 → ['Family', 'important', 'thing', ...]
```

`nltk.download('stopwords')`, `nltk.download('punkt_tab')`로 필요한 데이터를 받는다.


In [4]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt_tab')

txt = "Family is not an important thing. It's everything."
sw = set(stopwords.words('english'))

tokens = word_tokenize(txt)                          # 단어로 쪼개기
print("토큰화:", tokens)

filtered = [w for w in tokens if w.lower() not in sw]  # 불용어 제거
print("불용어 제거 후:", filtered)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


토큰화: ['Family', 'is', 'not', 'an', 'important', 'thing', '.', 'It', "'s", 'everything', '.']
불용어 제거 후: ['Family', 'important', 'thing', '.', "'s", 'everything', '.']


## 2부. 한국어 형태소 분석

영어는 띄어쓰기로 단어가 나뉜다. 하지만 **한국어는 다르다.**

> "공부한" = "공부"(명사) + "하"(동사) + "ㄴ"(어미)

한국어는 하나의 어절에 여러 형태소가 붙는다(교착어). 그래서 **띄어쓰기만으론 부족**하고,
**형태소 분석기**로 쪼개야 한다. 한국어 NLP 라이브러리 **KoNLPy**를 쓴다.

### 2-1. KoNLPy 설치

`konlpy`는 자바 기반이라 Colab에서 설치가 필요하다.


In [5]:
!pip install konlpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 43.5 MB/s eta 0:00:00


### 2-2. Okt — 트위터 형태소 분석기

**Okt**(Open Korean Text)는 빠르고 실용적인 분석기다. 세 가지 기능:

- `morphs` : 형태소로 쪼개기
- `pos` : 형태소 + **품사 태그** (Noun, Verb, Josa 등)
- `nouns` : **명사만** 뽑기

명사만 뽑는 기능이 뒤의 "헌법 명사 Top10" 실습에서 핵심이 된다.


In [6]:
from konlpy.tag import Okt

okt = Okt()
sent = "열심히 공부한 당신, 더 열심히 해 보아요~!!!"
print("형태소:", okt.morphs(sent))
print("품사  :", okt.pos(sent))
print("명사  :", okt.nouns(sent))


형태소: ['열심히', '공부', '한', '당신', ',', '더', '열심히', '해', '보아', '요', '~!!!']
품사  : [('열심히', 'Adverb'), ('공부', 'Noun'), ('한', 'Josa'), ('당신', 'Noun'), (',', 'Punctuation'), ('더', 'Noun'), ('열심히', 'Adverb'), ('해', 'Noun'), ('보아', 'Noun'), ('요', 'Josa'), ('~!!!', 'Punctuation')]
명사  : ['공부', '당신', '더', '해', '보아']


### 2-3. Kkma — 꼬꼬마 형태소 분석기

**Kkma**(꼬꼬마)는 서울대에서 만든 분석기다. Okt보다 **더 잘게, 더 정밀하게** 쪼갠다.
대신 느리다. 품사 태그 체계도 더 상세하다 (`NNG`=일반명사, `NNP`=고유명사 등).

**Okt vs Kkma:**
- Okt : 빠르고 실용적, 신조어·구어체에 강함
- Kkma : 정밀하고 상세, 학술적 분석에 적합 (대신 느림)

같은 문장을 두 분석기로 비교해본다.


In [7]:
from konlpy.tag import Kkma

kkma = Kkma()
print("형태소:", kkma.morphs(sent))
print("품사  :", kkma.pos(sent))
print("명사  :", kkma.nouns(sent))


형태소: ['열심히', '공부', '하', 'ㄴ', '당신', ',', '더', '열심히', '하', '어', '보', '아요', '~', '!!!']
품사  : [('열심히', 'MAG'), ('공부', 'NNG'), ('하', 'XSV'), ('ㄴ', 'ETD'), ('당신', 'NP'), (',', 'SP'), ('더', 'MAG'), ('열심히', 'MAG'), ('하', 'VV'), ('어', 'ECS'), ('보', 'VXV'), ('아요', 'EFN'), ('~', 'SO'), ('!!!', 'SW')]
명사  : ['공부', '당신']


## 3부. 실습 — 헌법에서 명사 Top 10

배운 걸 실전에 쓴다. **대한민국 헌법 전문에서 가장 많이 나온 명사 10개**를 뽑는다.

과정:
1. 헌법 텍스트 파일을 읽는다
2. 문장 단위로 나눈다 (`sent_tokenize`)
3. 각 문장을 Kkma로 품사 분석
4. **일반명사(`NNG`)만** 골라 개수를 센다
5. 가장 많은 10개 출력

> ⚠️ 이 셀은 `대한민국헌법.txt`가 Drive에 있어야 실행된다.
> 여기서는 `MyDrive/Colab Notebooks/` 폴더에 파일을 올려두고 마운트했다. (경로는 각자 환경에 맞게 `path`를 수정)
> (파일이 없으면 이 파트만 건너뛰어도 앞의 전처리 학습엔 지장 없다.)

In [8]:
from google.colab import drive
drive.mount('/content/gdrive')


Mounted at /content/gdrive


In [9]:
from collections import Counter

path = '/content/gdrive/MyDrive/Colab Notebooks/대한민국헌법.txt'
with open(path, encoding='utf-8') as f:
    doc = f.read()

wrdcnt = Counter()
for sent in nltk.sent_tokenize(doc):        # 문장 단위로 분리
    for word, tag in kkma.pos(sent):        # 형태소 + 품사
        if tag == 'NNG':                     # 일반명사만
            wrdcnt[word] += 1

print("명사 Top 10:")
for word, cnt in wrdcnt.most_common(10):
    print(f"  {word}: {cnt}회")


명사 Top 10:
  법률: 121회
  대통령: 84회
  국가: 73회
  헌법: 69회
  국민: 69회
  국회: 55회
  때: 55회
  회의: 42회
  필요: 31회
  위원: 31회


## 정리 — 텍스트 전처리

실행 결과로 확인한 것들:

**영어**

| 기법 | 예시 결과 | 특징 |
|---|---|---|
| **표제어(Lemma)** | 동사로: `dies→die, has→have, going→go` | 사전 기반, 진짜 단어. **품사를 알려줘야 정확** |
| **어간(Porter)** | `policy→polici, organization→organ` | 규칙 기반, 보수적 |
| **어간(Lancaster)** | `organization→org, love→lov` | 규칙 기반, 더 공격적으로 자름 |
| **불용어 제거** | `is, not, an, It` → 제거 | 흔한 단어를 걷어내 의미어만 남김 |

- **표제어 vs 어간**: 표제어는 사전을 찾아 진짜 단어로(느림·정확), 어간은 규칙으로 뭉텅 자름(빠름·거침).
  정확도가 중요하면 표제어, 속도·대량이면 어간.

**한국어 (KoNLPy)** — 같은 문장 `"열심히 공부한 당신, 더 열심히 해 보아요~!!!"`

| 분석기 | 명사 추출 결과 | 특징 |
|---|---|---|
| **Okt** | `['공부', '당신', '더', '해', '보아']` | 빠르고 실용적, 신조어·구어체에 강함 |
| **Kkma** | `['공부', '당신']` | 정밀·엄격, 진짜 명사만 골라냄 (느림) |

- 한국어는 교착어라 **띄어쓰기만으론 부족**하고 형태소 분석이 필수다.
- Okt는 "더/해/보아"까지 명사로 봤지만, Kkma는 진짜 명사 2개만 골랐다. **분석기마다 결과가 다르다.**

**실습 — 헌법 명사 Top 10**

```
법률: 121   대통령: 84   국가: 73   헌법: 69   국민: 69
국회: 55   때: 55   회의: 42   필요: 31   위원: 31
```

- Kkma로 일반명사(`NNG`)만 세어 `Counter.most_common(10)`으로 뽑았다.
- 헌법 문서답게 **법률·대통령·국가·헌법·국민** 같은 국가·법 관련 명사가 상위권.
- 이게 전처리의 실전 모습이다 — 원문을 형태소로 쪼개고, 원하는 품사만 골라, 통계를 낸다.

---

| 개념 | 핵심 |
|---|---|
| **Lemmatization** | 사전 기반 기본형. 품사(`'v'`)를 주면 정확 |
| **Stemming** | 규칙 기반 어미 자르기. Porter(보수) vs Lancaster(공격) |
| **Stopwords** | 의미 없는 흔한 단어 제거 |
| **KoNLPy** | 한국어 형태소 분석. Okt(빠름) vs Kkma(정밀) |
| **품사 태그** | `NNG`(일반명사)만 골라내는 식으로 활용 |

**다음 노트북(10)** 에서는 전처리한 단어를 **의미가 담긴 벡터로** 바꾼다 —
**Word2Vec**의 원리를 2차원으로 시각화하고(king·queen·man·woman),
영화 리뷰 데이터셋(IMDB)으로 실전 텍스트 분류를 준비한다.